# Phase 1: retained-data audit and collection provenance

Run from the repository root or Notebooks. Requires pandas and openpyxl. Original source text and labels are read-only. Duplicate checks use lowercase and collapsed whitespace only; no punctuation stripping, stemming or automatic deletion. Duplicate excess rows are counted beyond the first occurrence, while flags show every affected row. These checks concern the retained dataset, not historical cleaning removals. Collection claims below are unresolved report claims, not verified counts. This phase will be incorporated into the final combined notebook.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data/CPC 353 Dataset.xlsx").exists())
SOURCE = ROOT / "Data/CPC 353 Dataset.xlsx"
OUT = ROOT / "Outputs/Tables"
source_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
df = pd.read_excel(SOURCE, sheet_name="Self-Collected Data", header=3, usecols="A:M", keep_default_na=False)
populated = df.apply(lambda col: col.astype(str).str.strip().ne("")).any(axis=1)
df.insert(0, "source_row", range(5, len(df) + 5))
df = df.loc[populated].copy()

# Comparison keys only: original text, punctuation and emojis remain unchanged.
def normalise_for_comparison(text):
    return " ".join(str(text).lower().split())

def audit_records(frame):
    text = frame["Review Text"].astype(str)
    keys = text.map(normalise_for_comparison)
    nonblank = keys.ne("")
    ids = frame["Review ID"].astype(str).str.strip()
    flags = []
    masks = {
        "Blank review text": ~nonblank,
        "Blank review ID": ids.eq(""),
        "Duplicate review ID": ids.ne("") & ids.duplicated(keep=False),
        "Exact duplicate text": nonblank & text.duplicated(keep=False),
        "Normalised duplicate text": nonblank & keys.duplicated(keep=False),
    }
    for issue, mask in masks.items():
        for idx in frame.index[mask]:
            flags.append({"source_row": frame.loc[idx, "source_row"], "review_id": frame.loc[idx, "Review ID"], "issue": issue, "original_text": text.loc[idx], "comparison_key": keys.loc[idx]})
    summary = {
        "retained_reviews_checked": len(frame),
        "blank_review_texts": int((~nonblank).sum()),
        "blank_review_ids": int(ids.eq("").sum()),
        "duplicate_id_excess_rows": int(ids[ids.ne("")].duplicated().sum()),
        "exact_duplicate_text_excess_rows": int(text[nonblank].duplicated().sum()),
        "normalised_duplicate_text_excess_rows": int(keys[nonblank].duplicated().sum()),
    }
    return pd.DataFrame(summary.items(), columns=["check", "count"]), pd.DataFrame(flags, columns=["source_row", "review_id", "issue", "original_text", "comparison_key"])

summary, flags = audit_records(df)
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / "phase1_data_checks.csv", index=False)
flags.to_csv(OUT / "phase1_review_flags.csv", index=False, encoding="utf-8-sig")
restaurants = df.groupby("Restaurant Location", sort=False).size().reset_index(name="retained_reviews")
restaurants.to_csv(OUT / "phase1_restaurant_counts.csv", index=False, encoding="utf-8-sig")
provenance = pd.DataFrame([
    ["Five main collection runs", "275", "Reported in shared report; unverified", "Run manifests/raw exports not found in inspected workspace or Git history"],
    ["Approximate original collection", "around 300", "Reported in shared report; unverified", "May be rounding or a different collection scope; cannot determine from available files"],
    ["Retained annotated reviews", str(len(df)), "Verified from source workbook", "Self-Collected Data sheet; populated rows only"],
    ["Historical duplicates removed", "", "Unknown", "A clean final dataset does not establish how many records were previously removed"],
    ["Historical empty reviews removed", "", "Unknown", "Requires collection/cleaning records"],
    ["Historical other exclusions", "", "Unknown", "Requires collection/cleaning records"],
], columns=["stage", "count_or_claim", "status", "evidence_or_limitation"])
provenance.to_csv(OUT / "phase1_collection_provenance.csv", index=False, encoding="utf-8-sig")
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == source_hash
print(summary.to_string(index=False))
print(restaurants.to_string(index=False))
print("Source workbook unchanged:", source_hash)


                                check  count
             retained_reviews_checked    197
                   blank_review_texts      0
                     blank_review_ids      0
             duplicate_id_excess_rows      0
     exact_duplicate_text_excess_rows      0
normalised_duplicate_text_excess_rows      0
          Restaurant Location  retained_reviews
    Ayam Gepuk Macet Chow Kit                48
Ayam Gepuk Artisan Alam Damai                44
          Ayam Gepuk Original                37
        Gepuklah by mingchuun                19
Ayam Gepuk Pak Gembus Bangsar                49
Source workbook unchanged: bd587042d3bd3d1c514c9bf7dfa41bbbe92de58d676b666aa9fb12afb3f54dae
